# Tutorial: Knowledge Graph Federado

In [1]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao Python path para permitir imports do knowledge_graph
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"✓ Diretório {project_root} adicionado ao Python path")
print(f"✓ Agora é possível importar módulos do knowledge_graph")

✓ Diretório /Users/jwcunha/Documents/COMPANIES/AB/repos/private/premium/phd/computer-vision-and-nlp/rag/KRAG/v-antgvt adicionado ao Python path
✓ Agora é possível importar módulos do knowledge_graph


In [2]:
from knowledge_graph.federator  import FederadorOntologias
from knowledge_graph.ingester   import IngesterMarkdown
from knowledge_graph.detector   import DetectorEntidadesNovas
from knowledge_graph.validator  import ValidadorOntologia
from knowledge_graph.visualizer import VisualizadorKG

## 0. Fedoração e Inicialização
Nesse exemplo inicial, utilizaremos a base local do rdflib para não depender do endpoint QLever, o que é ótimo para protótipos e validação de dados curtos.

In [3]:
federador = FederadorOntologias(use_local_rdflib=True, base_dir='..')
federador.carregar_todos_os_grafos()

Carregando e3value de /Users/jwcunha/Documents/COMPANIES/AB/repos/private/premium/phd/computer-vision-and-nlp/rag/KRAG/v-antgvt/knowledge_graph/ontologias/e3value_subset.ttl
Carregando rea de /Users/jwcunha/Documents/COMPANIES/AB/repos/private/premium/phd/computer-vision-and-nlp/rag/KRAG/v-antgvt/knowledge_graph/ontologias/rea_subset.ttl
Carregando vdml de /Users/jwcunha/Documents/COMPANIES/AB/repos/private/premium/phd/computer-vision-and-nlp/rag/KRAG/v-antgvt/knowledge_graph/ontologias/vdml_subset.ttl
Carregando kg de /Users/jwcunha/Documents/COMPANIES/AB/repos/private/premium/phd/computer-vision-and-nlp/rag/KRAG/v-antgvt/knowledge_graph/ontologias/kg_custom.ttl


## 1. Ingestão (Extração de Triplas com LLM)

In [4]:
import os
from dotenv import load_dotenv

# Importar módulos necessários (caso célula seja executada isoladamente)
from knowledge_graph.ingester import IngesterMarkdown

# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# IMPORTANTE: Configure OPENAI_API_KEY no arquivo .env na raiz do projeto
# Exemplo no .env:
#   OPENAI_API_KEY=sk-proj-sua_chave_aqui
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("⚠️ OPENAI_API_KEY não encontrada!")
    print("Crie um arquivo .env na raiz do projeto com:")
    print("  OPENAI_API_KEY=sk-proj-sua_chave_aqui")
    print("\nExtração de triplas não funcionará sem a chave.")
else:
    print(f"✓ OPENAI_API_KEY carregada do .env (primeiros 10 chars: {api_key[:10]}...)")
    os.environ["OPENAI_API_KEY"] = api_key

ingester = IngesterMarkdown("dados_exemplo/apl_textil_pe_parte1.md", base_dir='..')
chunks   = ingester.carregar_chunks()
triplas  = []
for chunk in chunks:
    try: 
        triplas.extend(ingester.extrair_triplas(chunk))
    except Exception as e:
        print(f"Erro ao processar chunk: {e}")
        print("Verifique se OPENAI_API_KEY está configurada corretamente.")

if triplas:
    triplas_mapeadas = [ingester.mapear_para_ontologia(t) for t in triplas]
    federador.inserir_triplas(triplas_mapeadas)
    print(f"✓ {len(triplas_mapeadas)} triplas extraídas e inseridas.")
else:
    print("⚠️ Nenhuma tripla extraída. Verifique a configuração da API key.")

✓ OPENAI_API_KEY carregada do .env (primeiros 10 chars: sk-proj-Ud...)
Inseridas 6 triplas no grafo local.
✓ 6 triplas extraídas e inseridas.


## 2. Detecção de Candidatos


In [5]:
detector   = DetectorEntidadesNovas(federador)
candidatos = detector.identificar_e_marcar(triplas_mapeadas)
relatorio  = detector.gerar_relatorio_candidatos()
print("---- Relatório de Candidaturas ----")
print(relatorio)

Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
Inseridas 3 triplas no grafo local.
---- Relatório de Candidaturas ----
                                             uri          nome_entidade
0                http://phd-cesar-rag/kg#Cliente                Cliente
1                http://phd-cesar-rag/kg#Produto                Produto
2             http://phd-cesar-rag/kg#Fornecedor             Fornecedor
3          http://phd-cesar-rag/kg#Matéria-prima          Matéria-prima
4                  http://phd-cesar-rag/kg#Valor                  Valor
5                http://phd-cesar-rag/kg#Empresa                Empresa
6  http://phd-cesar-rag/kg#P

## 2.4 Interface de Validação

In [6]:
validador = ValidadorOntologia(federador)
# Simulação de aprovação manual de um subconjunto de itens:
if not relatorio.empty:
    for idx, row in relatorio.iterrows():
        # Exemplo automático (Na prática, 'input' humano)
        print(f"Aprovando automaticamente {row['nome_entidade']} para simplificar...")
        validador.aprovar_entidade(row['uri'], aprovador="TutorialBot", tipo_ontologico="http://phd-cesar-rag/kg#Biotecnologia")
        
validador.exportar_nova_versao_ontologia(versao="1.1.0", caminho_saida="../knowledge_graph/ontologias/kg_custom_v1.1.0.ttl")

Aprovando automaticamente Cliente para simplificar...
Aprovando automaticamente Produto para simplificar...
Aprovando automaticamente Fornecedor para simplificar...
Aprovando automaticamente Matéria-prima para simplificar...
Aprovando automaticamente Valor para simplificar...
Aprovando automaticamente Empresa para simplificar...
Aprovando automaticamente Processos de produção para simplificar...
Aprovando automaticamente Processo de produção para simplificar...
Aprovando automaticamente Risco para simplificar...
Exportada nova ontologia v1.1.0 em ../knowledge_graph/ontologias/kg_custom_v1.1.0.ttl


## 3. Visualização Interativa


In [7]:
viz = VisualizadorKG(federador)
viz.gerar_html_interativo("../kg_viz.html")
from IPython.display import IFrame
IFrame(src='../kg_viz.html', width='100%', height='600px')

Grafo visual salvo em ../kg_viz.html


In [8]:
import importlib
import knowledge_graph.visualizer
importlib.reload(knowledge_graph.visualizer)
from knowledge_graph.visualizer import VisualizadorKG

viz = VisualizadorKG(federador)
viz.gerar_html_interativo_instancias("../kg_viz_instancias.html")
from IPython.display import IFrame
IFrame(src='../kg_viz_instancias.html', width='100%', height='600px')

Grafo visual de instâncias salvo em ../kg_viz_instancias.html
